# Objective

Download the latest Metadata_tab files from ALLSTUDIES package from ImmPort using Method 2 from the tutorial notebook `immport-datasets.ipynb`.

This could help us to explore the datasets available in ImmPort more easily.

In [1]:
import sys
import os
import pandas as pd

# Set the Python path to the location of the directory containing "immport_download.py"
immport_download_code = "../../immport-data-download-tool/bin/" # Adjust this path as needed
sys.path.insert(0,immport_download_code)
os.chdir(immport_download_code)

import immport_download

In [2]:
# Configure parameters
user_name = input("ImmPort Username: ")
password = input("ImmPort Password: ")
download_directory = "../../output" # Note: Avoid using parent paths with spaces to prevent potential issues
data_directory = "../../data"


# Method 2

In [3]:
immport_download.download_file(user_name,password,
                                    "/ALLSTUDIES/ALLSTUDIES-DR59_Metadata_Tab.zip",download_directory)

Completed: 1520962K bytes transferred in 149 seconds
 (83138K bits/sec), in 1 file.


# Look for results data using Method 3

Using the ALLSTUDIES Metadata_tab files (`experiment.txt`), I identified several studies that have Olink results data available.

- SDY2011 (EXPERIMENT_ACCESSION: EXP34131, serum)
- SDY2189 (EXPERIMENT_ACCESSION: EXP36775)
- SDY2522 (EXPERIMENT_ACCESSION: EXP38528 for Olink Explore 3072, plasma)
- SDY2497 (EXPERIMENT_ACCESSION: EXP94378 for Inflammation panel)

In [5]:
# Request a token, then make API call, then load into Pandas's DataFrame
token = immport_download.request_immport_token(user_name, password)

In [13]:
# https://docs.immport.org/apidocumentation/shareddataapi/assayresults/

query_ls = [("SDY2011", "EXP34131"), # (target_study, target_experiment)
            ("SDY2189", "EXP36775"),
            ("SDY2522", "EXP38528"),
            ("SDY2497", "EXP94378"),
            ("SDY2011", "EXP34044") # This is a CyTOF experiment and I know it has results data
            ]
result_dict = {}

for target_study, target_experiment in query_ls:
    url = f"https://www.immport.org/data/query/result/filePath?studyAccession={target_study}&experimentAccession={target_experiment}"
    r = immport_download.api(url,token)
    result_dict[(target_study, target_experiment)] = pd.read_json(r)

/var/folders/s8/3jyy749s5dd02sr2jvqg53b40000gn/T/ipykernel_5953/1256369944.py:14: FutureWarning: Passing literal json to 'read_json' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  result_dict[(target_study, target_experiment)] = pd.read_json(r)
/var/folders/s8/3jyy749s5dd02sr2jvqg53b40000gn/T/ipykernel_5953/1256369944.py:14: FutureWarning: Passing literal json to 'read_json' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  result_dict[(target_study, target_experiment)] = pd.read_json(r)
/var/folders/s8/3jyy749s5dd02sr2jvqg53b40000gn/T/ipykernel_5953/1256369944.py:14: FutureWarning: Passing literal json to 'read_json' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  result_dict[(target_study, target_experiment)] = pd.read_json(r)
/var/folders/s8/3jyy749s5dd02sr2jvqg53b40000gn

In [8]:
result_dict.keys()

dict_keys([('SDY2011', 'EXP34131'), ('SDY2189', 'EXP36775'), ('SDY2522', 'EXP38528'), ('SDY2497', 'EXP94378')])

In [14]:
# Shape of each result DataFrame
for key in result_dict.keys():
    print(f"{key}: {result_dict[key].shape}")

('SDY2011', 'EXP34131'): (0, 0)
('SDY2189', 'EXP36775'): (0, 0)
('SDY2522', 'EXP38528'): (0, 0)
('SDY2497', 'EXP94378'): (0, 0)
('SDY2011', 'EXP34044'): (185, 37)


In [ ]:
# Try to search the studies (a broader search)


result_dict2 = {}

for target_study, _ in query_ls:
    url = f"https://www.immport.org/data/query/result/filePath?studyAccession={target_study}"
    r = immport_download.api(url,token)
    result_dict2[target_study] = pd.read_json(r)

/var/folders/s8/3jyy749s5dd02sr2jvqg53b40000gn/T/ipykernel_5953/3869240799.py:9: FutureWarning: Passing literal json to 'read_json' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  result_dict2[target_study] = pd.read_json(r)
/var/folders/s8/3jyy749s5dd02sr2jvqg53b40000gn/T/ipykernel_5953/3869240799.py:9: FutureWarning: Passing literal json to 'read_json' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  result_dict2[target_study] = pd.read_json(r)
/var/folders/s8/3jyy749s5dd02sr2jvqg53b40000gn/T/ipykernel_5953/3869240799.py:9: FutureWarning: Passing literal json to 'read_json' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  result_dict2[target_study] = pd.read_json(r)
/var/folders/s8/3jyy749s5dd02sr2jvqg53b40000gn/T/ipykernel_5953/3869240799.py:9: FutureWarning: Passing liter

In [ ]:
# Shape of each result DataFrame
for key in result_dict2.keys():
    shape = result_dict2[key].shape
    print(f"{key}: {shape}")
    if shape[0] > 0:
        unique_v = result_dict2[key]['fileDetail'].unique()
        print(f"Unique fileDetails: {unique_v}")

# Only CyTOF data found for all targeted studies

SDY2011: (185, 37)
Unique fileDetails: ['CyTOF result']
SDY2189: (6210, 37)
Unique fileDetails: ['CyTOF result' 'CyTOF workspace' 'Flow cytometry result'
 'Flow cytometry workspace']
SDY2522: (0, 0)
SDY2497: (80, 37)
Unique fileDetails: ['CyTOF result']


In [25]:
result_dict2["SDY2497"]

,filePathId,fileInfoId,fileDetail,filesizeBytes,fileName,originalFileName,sourceType,sourceAccession,experimentAccession,measurementTechnique,...,strain,armAccession,armName,ageEvent,ageEventSpecify,ageUnit,maxSubjectAge,minSubjectAge,subjectPhenotype,filePath
0,1050304_ES2150166_BS1394278,1050304,CyTOF result,152053144,5107_01112019_Flu2017_Study33_plate1_01112019 ...,5107_01112019_Flu2017_Study33_plate1_01112019 ...,EXPSAMPLE,ES2150166,EXP38445,CyTOF,...,NaN,ARM8523,Technical control,Age at enrollment,None,Years,90,0,NaN,/SDY2497/ResultFiles/CyTOF_result/5107_0111201...
1,1050305_ES2150167_BS1394279,1050305,CyTOF result,152053110,5110_01162019_Flu 2017_Study33_plate2_01162019...,5110_01162019_Flu 2017_Study33_plate2_01162019...,EXPSAMPLE,ES2150167,EXP38445,CyTOF,...,NaN,ARM8523,Technical control,Age at enrollment,None,Years,90,0,NaN,/SDY2497/ResultFiles/CyTOF_result/5110_0116201...
2,1050306_ES2150168_BS1394280,1050306,CyTOF result,152053142,5109_01282019_Flu 2017Study33_plate3_01282019 ...,5109_01282019_Flu 2017Study33_plate3_01282019 ...,EXPSAMPLE,ES2150168,EXP38445,CyTOF,...,NaN,ARM8523,Technical control,Age at enrollment,None,Years,90,0,NaN,/SDY2497/ResultFiles/CyTOF_result/5109_0128201...
3,1050307_ES2150169_BS1394210,1050307,CyTOF result,152053109,5110_01162019_Flu 2017_Study33_plate2_01162019...,5110_01162019_Flu 2017_Study33_plate2_01162019...,EXPSAMPLE,ES2150169,EXP38445,CyTOF,...,NaN,ARM8521,Arm 1,Other,"Demographic Form Date,2017-09,V01",Years,38,38,NaN,/SDY2497/ResultFiles/CyTOF_result/5110_0116201...
4,1050308_ES2150170_BS1394212,1050308,CyTOF result,152053116,5110_01162019_Flu 2017_Study33_plate2_01162019...,5110_01162019_Flu 2017_Study33_plate2_01162019...,EXPSAMPLE,ES2150170,EXP38445,CyTOF,...,NaN,ARM8521,Arm 1,Other,"Demographic Form Date,2017-09,V01",Years,29,29,NaN,/SDY2497/ResultFiles/CyTOF_result/5110_0116201...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,1054906_ES2154413_BS1394315,1054906,CyTOF result,155053663,6400_study33_plate4_01262021_study33_plate4 01...,6400_study33_plate4_01262021_study33_plate4 01...,EXPSAMPLE,ES2154413,EXP38501,CyTOF,...,NaN,ARM8522,Arm 2,Other,"Demographic Form Date,2018-10,V01",Years,78,78,NaN,/SDY2497/ResultFiles/CyTOF_result/6400_study33...
76,1054907_ES2154414_BS1394317,1054907,CyTOF result,155053660,6400_study33_plate4_01262021_study33_plate4 01...,6400_study33_plate4_01262021_study33_plate4 01...,EXPSAMPLE,ES2154414,EXP38501,CyTOF,...,NaN,ARM8521,Arm 1,Other,"Demographic Form Date,2018-09,V01",Years,24,24,NaN,/SDY2497/ResultFiles/CyTOF_result/6400_study33...
77,1054908_ES2154415_BS1394318,1054908,CyTOF result,155053661,6400_study33_plate4_01262021_study33_plate4 01...,6400_study33_plate4_01262021_study33_plate4 01...,EXPSAMPLE,ES2154415,EXP38501,CyTOF,...,NaN,ARM8521,Arm 1,Other,"Demographic Form Date,2018-10,V01",Years,25,25,NaN,/SDY2497/ResultFiles/CyTOF_result/6400_study33...
78,1054909_ES2154416_BS1394320,1054909,CyTOF result,155053659,6400_study33_plate4_01262021_study33_plate4 01...,6400_study33_plate4_01262021_study33_plate4 01...,EXPSAMPLE,ES2154416,EXP38501,CyTOF,...,NaN,ARM8521,Arm 1,Other,"Demographic Form Date,2018-10,V01",Years,40,40,NaN,/SDY2497/ResultFiles/CyTOF_result/6400_study33...


In [26]:
# Try another broader search to include Study data and Result data

result_dict3 = {}

for target_study, _ in query_ls:
    url = f"https://www.immport.org/data/query/api/study/file/{target_study}" # See "immport-datasets-SDY2011_2025-12-30.ipynb"

    r = immport_download.api(url,token)
    result_dict3[target_study] = pd.read_json(r)

/var/folders/s8/3jyy749s5dd02sr2jvqg53b40000gn/T/ipykernel_5953/3941440247.py:9: FutureWarning: Passing literal json to 'read_json' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  result_dict3[target_study] = pd.read_json(r)
/var/folders/s8/3jyy749s5dd02sr2jvqg53b40000gn/T/ipykernel_5953/3941440247.py:9: FutureWarning: Passing literal json to 'read_json' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  result_dict3[target_study] = pd.read_json(r)
/var/folders/s8/3jyy749s5dd02sr2jvqg53b40000gn/T/ipykernel_5953/3941440247.py:9: FutureWarning: Passing literal json to 'read_json' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  result_dict3[target_study] = pd.read_json(r)
/var/folders/s8/3jyy749s5dd02sr2jvqg53b40000gn/T/ipykernel_5953/3941440247.py:9: FutureWarning: Passing liter

In [29]:
# Shape of the results data
# studyFileType column unique values
# description column unique values
for key in result_dict3.keys():
    shape = result_dict3[key].shape
    print(f"{key}: {shape}")
    if shape[0] > 0:
        unique_v = result_dict3[key]['studyFileType'].unique()
        print(f"Unique studyFileType: {unique_v}")
        unique_desc = result_dict3[key]['description'].unique()
        print(f"Unique description: {unique_desc}")

SDY2011: (189, 6)
Unique studyFileType: ['Study Data']
Unique description: ['CyTOF fcs' 'Olink data' 'Olink patient samples metadata'
 'Olink control samples metadata' 'Olink Data']
SDY2189: (1, 6)
Unique studyFileType: ['Study Summary Description']
Unique description: ['Study description']
SDY2522: (7, 6)
Unique studyFileType: ['Study Data' 'Demographics' 'Study Summary Description']
Unique description: ['Expression of IL-6, IL-8, NFKB1, MMP8, IL1B, CASP8, IFI30, USP8, EREG, C-reactive protein, Fibrinogen, B-cell responses in PASC patients and controls; Serological reactivity against receptor binding domain, nucleocapsid, Serological anti nucleocapsid responses in inflammatory PASC (inflPASC) and non-inflammatory PASC patients (niPASC) cohorts.'
 'Feature importance (potency) scores for all data features based on 10,000 RF classificationsbetween post-acute sequelae of COVID-19 (PASC) patients and COVID-recovery (CR) controls groups.'
 'PASC symptom presentations between inflammatory P

In [31]:
result_dict3["SDY2522"]

,studyFileAccession,studyAccession,studyFileType,description,fileName,filePath
0,SFL29277,SDY2522,Study Data,"Expression of IL-6, IL-8, NFKB1, MMP8, IL1B, C...",PMID37452024_41467_2023_40012_MOESM7_ESM.xlsx,/SDY2522/StudyFiles/PMID37452024_41467_2023_40...
1,SFL29278,SDY2522,Study Data,Feature importance (potency) scores for all da...,PMID37452024_Supplementary Data 1.xlsx,/SDY2522/StudyFiles/PMID37452024_Supplementary...
2,SFL29279,SDY2522,Study Data,PASC symptom presentations between inflammator...,PMID37452024_Supplementary Data 2.xlsx,/SDY2522/StudyFiles/PMID37452024_Supplementary...
3,SFL29280,SDY2522,Study Data,Figures of correlation matrix of proteomics da...,PMID37452024_Supplementary Information.pdf,/SDY2522/StudyFiles/PMID37452024_Supplementary...
4,SFL29281,SDY2522,Demographics,Cohort level demographics of post-acute sequel...,PMID37452024_Patient data table.xlsx,/SDY2522/StudyFiles/PMID37452024_Patient data ...
5,SFL29284,SDY2522,Study Data,Publication in PDF format.,PMID37452024_publication.pdf,/SDY2522/StudyFiles/PMID37452024_publication.pdf
6,SFL73567,SDY2522,Study Summary Description,Completed curation of the study into SeroNet R...,PMID37452024_v1.3.5-02.xlsm,/SDY2522/StudyFiles/PMID37452024_v1.3.5-02.xlsm


In [ ]:
# Can we find the same result as result_dict3 using "https://www.immport.org/data/query/api/study/"
# https://docs.immport.org/apidocumentation/shareddataapi/study/

result_dict4 = {}

for target_study, _ in query_ls:
    url = f"https://www.immport.org/data/query/api/study/file/{target_study}"

    r = immport_download.api(url,token)
    result_dict4[target_study] = pd.read_json(r)

/var/folders/s8/3jyy749s5dd02sr2jvqg53b40000gn/T/ipykernel_5953/1638198877.py:10: FutureWarning: Passing literal json to 'read_json' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  result_dict4[target_study] = pd.read_json(r)
/var/folders/s8/3jyy749s5dd02sr2jvqg53b40000gn/T/ipykernel_5953/1638198877.py:10: FutureWarning: Passing literal json to 'read_json' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  result_dict4[target_study] = pd.read_json(r)
/var/folders/s8/3jyy749s5dd02sr2jvqg53b40000gn/T/ipykernel_5953/1638198877.py:10: FutureWarning: Passing literal json to 'read_json' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  result_dict4[target_study] = pd.read_json(r)
/var/folders/s8/3jyy749s5dd02sr2jvqg53b40000gn/T/ipykernel_5953/1638198877.py:10: FutureWarning: Passing l

In [34]:
result_dict4["SDY2522"]

,studyFileAccession,studyAccession,studyFileType,description,fileName,filePath
0,SFL29277,SDY2522,Study Data,"Expression of IL-6, IL-8, NFKB1, MMP8, IL1B, C...",PMID37452024_41467_2023_40012_MOESM7_ESM.xlsx,/SDY2522/StudyFiles/PMID37452024_41467_2023_40...
1,SFL29278,SDY2522,Study Data,Feature importance (potency) scores for all da...,PMID37452024_Supplementary Data 1.xlsx,/SDY2522/StudyFiles/PMID37452024_Supplementary...
2,SFL29279,SDY2522,Study Data,PASC symptom presentations between inflammator...,PMID37452024_Supplementary Data 2.xlsx,/SDY2522/StudyFiles/PMID37452024_Supplementary...
3,SFL29280,SDY2522,Study Data,Figures of correlation matrix of proteomics da...,PMID37452024_Supplementary Information.pdf,/SDY2522/StudyFiles/PMID37452024_Supplementary...
4,SFL29281,SDY2522,Demographics,Cohort level demographics of post-acute sequel...,PMID37452024_Patient data table.xlsx,/SDY2522/StudyFiles/PMID37452024_Patient data ...
5,SFL29284,SDY2522,Study Data,Publication in PDF format.,PMID37452024_publication.pdf,/SDY2522/StudyFiles/PMID37452024_publication.pdf
6,SFL73567,SDY2522,Study Summary Description,Completed curation of the study into SeroNet R...,PMID37452024_v1.3.5-02.xlsm,/SDY2522/StudyFiles/PMID37452024_v1.3.5-02.xlsm


In [42]:
result_dict4["SDY2011"]

,studyFileAccession,studyAccession,studyFileType,description,fileName,filePath
0,SFL22034,SDY2011,Study Data,CyTOF fcs,200422_HIMC15_PICR_MDIPA_Subj_089e98eaT1_debc.fcs,/SDY2011/StudyFiles/200422_HIMC15_PICR_MDIPA_S...
1,SFL22035,SDY2011,Study Data,CyTOF fcs,200422_HIMC15_PICR_MDIPA_Subj_29703e49T1_debc.fcs,/SDY2011/StudyFiles/200422_HIMC15_PICR_MDIPA_S...
2,SFL22036,SDY2011,Study Data,CyTOF fcs,200422_HIMC15_PICR_MDIPA_Subj_2d3ac920T1_debc.fcs,/SDY2011/StudyFiles/200422_HIMC15_PICR_MDIPA_S...
3,SFL22037,SDY2011,Study Data,CyTOF fcs,200422_HIMC15_PICR_MDIPA_Subj_34518e80T1_debc.fcs,/SDY2011/StudyFiles/200422_HIMC15_PICR_MDIPA_S...
4,SFL22038,SDY2011,Study Data,CyTOF fcs,200422_HIMC15_PICR_MDIPA_Subj_416ded86T1_debc.fcs,/SDY2011/StudyFiles/200422_HIMC15_PICR_MDIPA_S...
...,...,...,...,...,...,...
184,SFL22218,SDY2011,Study Data,CyTOF fcs,200616_MDIPAa4_Subj_c06f90b7T8_debc.fcs,/SDY2011/StudyFiles/200616_MDIPAa4_Subj_c06f90...
185,SFL22219,SDY2011,Study Data,Olink data,Chen_MBAA_Results.xlsx,/SDY2011/StudyFiles/Chen_MBAA_Results.xlsx
186,SFL22220,SDY2011,Study Data,Olink patient samples metadata,olink_36plt_202012_pt_meta.csv,/SDY2011/StudyFiles/olink_36plt_202012_pt_meta...
187,SFL22221,SDY2011,Study Data,Olink control samples metadata,olink_36plt_20201030_ctrlsamples_meta.csv,/SDY2011/StudyFiles/olink_36plt_20201030_ctrls...


In [35]:
# Shape of the results data for SDY2011 EXP34131

for key in result_dict4.keys():
    shape = result_dict4[key].shape
    print(f"{key}: {shape}")
    if shape[0] > 0:
        unique_v = result_dict4[key]['studyFileType'].unique()
        print(f"Unique studyFileType: {unique_v}")
        unique_desc = result_dict4[key]['description'].unique()
        print(f"Unique description: {unique_desc}")

# YES, it seems we can get the same results using either of the two APIs

SDY2011: (189, 6)
Unique studyFileType: ['Study Data']
Unique description: ['CyTOF fcs' 'Olink data' 'Olink patient samples metadata'
 'Olink control samples metadata' 'Olink Data']
SDY2189: (1, 6)
Unique studyFileType: ['Study Summary Description']
Unique description: ['Study description']
SDY2522: (7, 6)
Unique studyFileType: ['Study Data' 'Demographics' 'Study Summary Description']
Unique description: ['Expression of IL-6, IL-8, NFKB1, MMP8, IL1B, CASP8, IFI30, USP8, EREG, C-reactive protein, Fibrinogen, B-cell responses in PASC patients and controls; Serological reactivity against receptor binding domain, nucleocapsid, Serological anti nucleocapsid responses in inflammatory PASC (inflPASC) and non-inflammatory PASC patients (niPASC) cohorts.'
 'Feature importance (potency) scores for all data features based on 10,000 RF classificationsbetween post-acute sequelae of COVID-19 (PASC) patients and COVID-recovery (CR) controls groups.'
 'PASC symptom presentations between inflammatory P

In [ ]:
# Can we obtain the same use `filePath/{StudyAccession}` instead of `file/{StudyAccession}`

result_dict5 = {}

for target_study, _ in query_ls:
    url = f"https://www.immport.org/data/query/api/study/filePath/{target_study}"

    r = immport_download.api(url,token)
    result_dict5[target_study] = pd.read_json(r)

# The results are not the same

/var/folders/s8/3jyy749s5dd02sr2jvqg53b40000gn/T/ipykernel_5953/2083024393.py:9: FutureWarning: Passing literal json to 'read_json' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  result_dict5[target_study] = pd.read_json(r)
/var/folders/s8/3jyy749s5dd02sr2jvqg53b40000gn/T/ipykernel_5953/2083024393.py:9: FutureWarning: Passing literal json to 'read_json' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  result_dict5[target_study] = pd.read_json(r)
/var/folders/s8/3jyy749s5dd02sr2jvqg53b40000gn/T/ipykernel_5953/2083024393.py:9: FutureWarning: Passing literal json to 'read_json' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  result_dict5[target_study] = pd.read_json(r)
/var/folders/s8/3jyy749s5dd02sr2jvqg53b40000gn/T/ipykernel_5953/2083024393.py:9: FutureWarning: Passing liter

In [39]:
# Shape of the results data 

for key in result_dict5.keys():
    shape = result_dict5[key].shape
    print(f"{key}: {shape}")
    if shape[0] > 0:
        unique_v = result_dict5[key]['fileType'].unique()
        print(f"Unique fileType: {unique_v}")
        

SDY2011: (393, 7)
Unique fileType: ['study_file' 'protocol_file' 'result_file' 'release_file' 'archive_file']
SDY2189: (522, 7)
Unique fileType: ['study_file' 'protocol_file' 'result_file' 'release_file' 'archive_file']
SDY2522: (24, 7)
Unique fileType: ['study_file' 'protocol_file' 'release_file' 'archive_file']
SDY2497: (98, 7)
Unique fileType: ['protocol_file' 'result_file' 'release_file' 'archive_file']


In [38]:
result_dict5["SDY2522"]

,fileDetailsId,generatedMD5,studyAccession,fileType,fileAccession,fileName,path
0,359656,8f262db864f9a48e4cc2c30223d25bfc,SDY2522,study_file,SFL29280,PMID37452024_Supplementary Information.pdf,SDY2522/StudyFiles/PMID37452024_Supplementary ...
1,359657,d40f5679d44feab3f7ada1fc51c4bd24,SDY2522,study_file,SFL29277,PMID37452024_41467_2023_40012_MOESM7_ESM.xlsx,SDY2522/StudyFiles/PMID37452024_41467_2023_400...
2,359659,9615452d163bdd0581dc4df6759c3f40,SDY2522,study_file,SFL29279,PMID37452024_Supplementary Data 2.xlsx,SDY2522/StudyFiles/PMID37452024_Supplementary ...
3,359665,1d36248cd80081a25409218a418a134e,SDY2522,study_file,SFL29278,PMID37452024_Supplementary Data 1.xlsx,SDY2522/StudyFiles/PMID37452024_Supplementary ...
4,359669,db49865967e033e2265701941a9108d6,SDY2522,study_file,SFL29284,PMID37452024_publication.pdf,SDY2522/StudyFiles/PMID37452024_publication.pdf
5,359671,a51d6600dfe4ec28ea43a8a079368f95,SDY2522,study_file,SFL29282,PMID37452024_v1.3.5.xlsm,SDY2522/StudyFiles/PMID37452024_v1.3.5.xlsm
6,359673,4f9dc2af3036e7debb23a8658ddf95e2,SDY2522,study_file,SFL29281,PMID37452024_Patient data table.xlsx,SDY2522/StudyFiles/PMID37452024_Patient data t...
7,359675,71f470c396a2cc2205d36ce7d7021f59,SDY2522,protocol_file,PTL11676,PMID37452024_protocol-01.PTL11676.txt,SDY2522/Protocols/PMID37452024_protocol-01.PTL...
8,550773,bce11090129543ed5a37c11c4a4ece14,SDY2522,study_file,SFL30237,PMID37452024_JSON-v1.json,SDY2522/StudyFiles/PMID37452024_JSON-v1.json
9,550774,c15e86b05935f1d43f7a5c4b919aadf5,SDY2522,study_file,SFL30238,PMID37452024_v1.3.5-v1.xlsm,SDY2522/StudyFiles/PMID37452024_v1.3.5-v1.xlsm


In [41]:
result_dict5["SDY2011"]

,fileDetailsId,generatedMD5,studyAccession,fileType,fileAccession,fileName,path
0,8253,3c7e8235b56c33ab75c34acddca1704e,SDY2011,study_file,SFL22381,olink_36plt_202012_pt_meta_12052022_removed.csv,SDY2011/StudyFiles/olink_36plt_202012_pt_meta_...
1,9871,ea63c7c3a48912ec0c9833337b7048f3,SDY2011,study_file,SFL22034,200422_HIMC15_PICR_MDIPA_Subj_089e98eaT1_debc.fcs,SDY2011/StudyFiles/200422_HIMC15_PICR_MDIPA_Su...
2,9872,b3f5f088db723b389e7ee24ee4304bad,SDY2011,study_file,SFL22035,200422_HIMC15_PICR_MDIPA_Subj_29703e49T1_debc.fcs,SDY2011/StudyFiles/200422_HIMC15_PICR_MDIPA_Su...
3,9873,51126349772009bb5c14534601d305b9,SDY2011,study_file,SFL22036,200422_HIMC15_PICR_MDIPA_Subj_2d3ac920T1_debc.fcs,SDY2011/StudyFiles/200422_HIMC15_PICR_MDIPA_Su...
4,9874,18800883a566f8e86e63c3b8cad74a03,SDY2011,study_file,SFL22037,200422_HIMC15_PICR_MDIPA_Subj_34518e80T1_debc.fcs,SDY2011/StudyFiles/200422_HIMC15_PICR_MDIPA_Su...
...,...,...,...,...,...,...,...
388,856102,d756804ce3c93b06242d1d39d943cc9b,SDY2011,archive_file,SDY2011-DR53.zip,SDY2011-DR53.zip,SDY2011/archive/SDY2011-DR53.zip
389,856103,e9a2aeaef93d1b5fb40b7f034d6c14ce,SDY2011,archive_file,SDY2011-DR49.zip,SDY2011-DR49.zip,SDY2011/archive/SDY2011-DR49.zip
390,856104,0c9455098c70769ecfb103627f4a7747,SDY2011,archive_file,SDY2011-DR50.zip,SDY2011-DR50.zip,SDY2011/archive/SDY2011-DR50.zip
391,856105,1200c564fe4221124e3d4ab7d5a16006,SDY2011,archive_file,SDY2011-DR48.zip,SDY2011-DR48.zip,SDY2011/archive/SDY2011-DR48.zip


In [ ]:
# Download specific files from SDY2522
    # /SDY2522/StudyFiles/PMID37452024_41467_2023_40012_MOESM7_ESM.xlsx
    # /SDY2522/StudyFiles/PMID37452024_Patient data table.xlsx

immport_download.download_file(user_name,password,
    "/SDY2522/StudyFiles/PMID37452024_41467_2023_40012_MOESM7_ESM.xlsx", download_directory)
immport_download.download_file(user_name,password,
    "/SDY2522/StudyFiles/PMID37452024_Patient data table.xlsx", download_directory)

# => There are some markers (it seems like NPX values) in the first file. The data is link to the figures in the paper.

Completed: 57K bytes transferred in 0 seconds
 (931K bits/sec), in 1 file.
Completed: 15K bytes transferred in 0 seconds
 (193K bits/sec), in 1 file.
